# Building a RAG

## Installing and prep

In [1]:
!brew install ollama

==> Downloading https://formulae.brew.sh/api/formula.jws.json
==> Downloading https://formulae.brew.sh/api/cask.jws.json
To reinstall 0.11.4, run:
  brew reinstall ollama


In [2]:
!ollama pull llama3:8b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling 6a0746a1ec1a: 100% ▕██████████████████▏ 4.7 GB                         
pulling 4fa551d4f938: 100% ▕██████████████████▏  12 KB                         
pulling 8ab4849b038c: 100% ▕██████████████████▏  254 B                         
pulling 577073ffcc6c: 100% ▕██████████████████▏  110 B                         
pulling 3f8eb4da87fa: 100% ▕██████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 


In [6]:
!mkdir -p data
# Manually download “Complete Works of William Shakespeare” (UTF-8 text) into data/shakespeare.txt
# e.g., from Project Gutenberg (ID often 100). Keep it offline after download.


In [1]:
!pip install faiss-cpu sentence-transformers ollama


## Cleaning

In [2]:
from pathlib import Path
import re, json

raw = Path("data/shakespeare.txt").read_text(encoding="utf-8", errors="ignore")

# crude split by titles in ALL CAPS
blocks = re.split(r"\n\s*\n(?=[A-Z][A-Z \-\']+\n)", raw)
docs = []
for b in blocks:
    lines = [l.strip() for l in b.splitlines() if l.strip()]
    if not lines: 
        continue
    title = lines[0] if lines[0].isupper() else "UNKNOWN"
    text  = " ".join(lines[1:] if title!="UNKNOWN" else lines)
    if len(text) < 500: 
        continue
    docs.append({"play": title.title(), "text": text})

print(f"Loaded {len(docs)} plays")


Loaded 87 plays


## Chunking

In [4]:
CHUNK_SIZE = 900
OVERLAP    = 200

def chunk_text(play, text):
    chunks = []
    for i in range(0, max(len(text)-CHUNK_SIZE, 0)+1, CHUNK_SIZE-OVERLAP):
        chunk = text[i:i+CHUNK_SIZE]
        chunks.append({"play": play, "start": i, "end": i+CHUNK_SIZE, "text": chunk})
    if not chunks:
        chunks.append({"play": play, "start": 0, "end": len(text), "text": text})
    return chunks

all_chunks = []
for d in docs:
    all_chunks.extend(chunk_text(d["play"], d["text"]))

len(all_chunks)


7083

## Building Index

In [5]:
import faiss, numpy as np
from sentence_transformers import SentenceTransformer

EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(EMB_MODEL)

embs = model.encode([c["text"] for c in all_chunks], 
                    batch_size=64, 
                    show_progress_bar=True, 
                    normalize_embeddings=True)

embs = np.asarray(embs).astype("float32")

d = embs.shape[1]
index = faiss.IndexFlatIP(d)  # cosine if normalized
index.add(embs)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2006.98it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 111/111 [00:22<00:00,  5.03it/s]


## Retreival

In [6]:
def retrieve(query, k=5):
    qe = model.encode([query], normalize_embeddings=True)
    D, I = index.search(np.asarray(qe).astype("float32"), k)
    hits = [{"score": float(D[0][i]), **all_chunks[I[0][i]]} for i in range(k)]
    return hits


## Prompt Builder

In [18]:
def build_prompt(question, hits):
    context = "\n\n---\n\n".join(
        f"[{i+1}] {h['play']} (chars {h['start']}-{h['end']}):\n{h['text'][:800]}"
        for i,h in enumerate(hits)
    )
    system = (
        "Answer strictly using GenZ slang."
        "Cite sources like [1], [2]. "
        "If the answer isn't in the context, say you don't know."
    )
    user = f"Question: {question}\n\nContext:\n{context}\n\nAnswer:"
    return system, user


In [19]:
import ollama

def ask_llm(system, user, model_name="llama3:8b"):
    prompt = f"<<SYS>>\n{system}\n<</SYS>>\n{user}"
    r = ollama.chat(model=model_name, messages=[{"role":"user","content":prompt}], stream=False)
    return r["message"]["content"]


## Testing

In [23]:
question = "How did Dumbledore die?"
hits = retrieve(question, k=5)

system, user = build_prompt(question, hits)
answer = ask_llm(system, user)

print("Retrieved from plays:")
for h in hits:
    print("-", h["play"], f"(score={h['score']:.3f})")

print("\nLLM Answer:\n", answer)


Retrieved from plays:
- Induction (score=0.382)
- Dramatis Personae (score=0.330)
- Dramatis Personae (score=0.322)
- Dramatis Personae (score=0.320)
- Prologue (score=0.309)

LLM Answer:
 Yaaas, I got this!

Dumbledore didn't actually die in the Harry Potter series. He sacrificed himself to save his dear friend Harry in the book "Harry Potter and the Half-Blood Prince" by J.K. Rowling [1]. He went back in time and appeared to himself as a younger man, thus creating a paradox and ultimately leaving his physical body behind. This is why he didn't actually die, but rather, his physical form passed away.

So, to answer your question, Dumbledore didn't actually die. He's just, like, totally gone, you know?
